# 02 — Transformer Decoder-Only para Tutor de Música

In [ ]:
# Instale apenas se necessário.
import sys
import subprocess

def pip_install(*packages):
    subprocess.check_call([sys.executable, "-m", "pip", "install", *packages])

# pip_install("torch", "tokenizers>=0.15.0", "tqdm", "numpy")

In [2]:
import random
import re
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from typing import Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tokenizers import Tokenizer
from tqdm.auto import tqdm

c:\Users\jgmda\Documents\GitHub\Doutorado\DeepLearningLLMs\TrabalhoFinal\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Configuração

`block_size` continua sendo o limite máximo de contexto do modelo.

In [3]:
@dataclass
class Config:
    tokenizer_path: str = "tokenizer/tokenizer.json"
    data_dir: str = "data"
    checkpoint_dir: str = "checkpoints"

    block_size: int = 256
    batch_size: int = 16
    max_steps: int = 400        # era 5000 — reduzido para smoke test
    eval_interval: int = 25     # era 100 — reduzido para ver a curva train/val mais cedo
    eval_batches: int = 50

    train_ratio: float = 0.80
    val_ratio: float = 0.10
    test_ratio: float = 0.10

    n_layer: int = 4
    n_head: int = 4
    n_embd: int = 256
    dropout: float = 0.1

    learning_rate: float = 3e-4
    weight_decay: float = 0.1
    grad_clip: float = 1.0

cfg = Config()

assert abs(cfg.train_ratio + cfg.val_ratio + cfg.test_ratio - 1.0) < 1e-6
assert cfg.n_embd % cfg.n_head == 0

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

if device == "cuda":
    torch.set_float32_matmul_precision("high")

Device: cuda


## 2. Carregar tokenizer

In [4]:
tokenizer_path = Path(cfg.tokenizer_path)

if not tokenizer_path.exists():
    raise FileNotFoundError(
        f"Tokenizer não encontrado em {tokenizer_path}. "
        "Execute primeiro o notebook do tokenizer."
    )

tokenizer = Tokenizer.from_file(str(tokenizer_path))

PAD_ID = tokenizer.token_to_id("<pad>")
BOS_ID = tokenizer.token_to_id("<bos>")
EOS_ID = tokenizer.token_to_id("<eos>")

required_tokens = [
    "<pad>", "<bos>", "<eos>", "<unk>",
    "<pergunta>", "<resposta>", "<topico>", "<conteudo>",
    "<exercicio>", "<resolucao>", "<gabarito>"
]

missing = [t for t in required_tokens if tokenizer.token_to_id(t) is None]

if missing:
    raise ValueError(f"Tokens especiais ausentes no tokenizer: {missing}")

vocab_size = tokenizer.get_vocab_size()

print("Vocab size:", vocab_size)
print("PAD/BOS/EOS:", PAD_ID, BOS_ID, EOS_ID)

Vocab size: 4488
PAD/BOS/EOS: 0 1 2


## 3. Carregar documentos `.txt`

In [5]:
data_dir = Path(cfg.data_dir)
files = sorted(data_dir.glob("*.txt"))

if not files:
    raise FileNotFoundError("Nenhum arquivo .txt encontrado em ./data.")

documents = []

for path in files:
    text = path.read_text(encoding="utf-8", errors="ignore").strip()

    if text:
        documents.append({
            "name": path.name,
            "text": text,
            "chars": len(text),
        })

if not documents:
    raise ValueError("Os arquivos .txt encontrados estão vazios.")

print("Documentos carregados:", len(documents))
print("Caracteres totais:", f'{sum(d["chars"] for d in documents):,}')

for d in documents:
    print(f'- {d["name"]}: {d["chars"]:,} caracteres')

Documentos carregados: 1
Caracteres totais: 186,587
- teste.txt: 186,587 caracteres


## 4. Separar amostras semânticas

Esta é a principal mudança.

O notebook separa o corpus em unidades completas:

- `<topico>` + `<conteudo>`
- `<pergunta>` + `<resposta>`
- `<exercicio>` + `<resolucao>` + `<gabarito>`

Depois divide essas amostras entre treino, validação e teste.

In [6]:
## 4. Separar amostras semânticas (tageadas + texto livre)

START_PATTERN = re.compile(
    r"(?=<(?:topico|pergunta|exercicio)>\s*)",
    flags=re.IGNORECASE
)

# Tamanho-alvo (em caracteres) de cada "chunk" de texto livre.
# Não é um corte rígido: tentamos fechar no fim de um parágrafo.
FREE_TEXT_TARGET_CHARS = 800
FREE_TEXT_MIN_CHARS = 150  # chunks menores que isso são descartados (ruído)


def _chunk_free_text(text: str, target_chars: int = FREE_TEXT_TARGET_CHARS) -> list[str]:
    """
    Divide um trecho de texto corrido (sem tags) em pedaços por parágrafo,
    agrupando parágrafos consecutivos até atingir ~target_chars.
    """
    paragraphs = [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]

    chunks = []
    current = []
    current_len = 0

    for p in paragraphs:
        current.append(p)
        current_len += len(p)

        if current_len >= target_chars:
            chunks.append("\n\n".join(current))
            current = []
            current_len = 0

    if current:
        chunks.append("\n\n".join(current))

    return [c for c in chunks if len(c) >= FREE_TEXT_MIN_CHARS]


def split_semantic_samples(text: str) -> list[dict]:
    """
    Separa o documento em amostras completas.

    Retorna uma lista de dicts: {"text": ..., "tagged": bool}

    - Blocos que começam com <topico>, <pergunta> ou <exercicio> são tratados
      como amostras tageadas (como antes).
    - Qualquer texto fora desses blocos (incluindo documentos sem tag alguma)
      é tratado como texto livre e dividido em chunks por parágrafo.
    """
    text = text.strip()

    if not text:
        return []

    pieces = START_PATTERN.split(text)
    samples = []

    for piece in pieces:
        piece = piece.strip()

        if not piece:
            continue

        if (
            piece.startswith("<topico>")
            or piece.startswith("<pergunta>")
            or piece.startswith("<exercicio>")
        ):
            samples.append({"text": piece, "tagged": True})
        else:
            # Texto sem tag de abertura: trata como texto livre.
            for chunk in _chunk_free_text(piece):
                samples.append({"text": chunk, "tagged": False})

    return samples


def classify_sample(sample: dict) -> str:
    text = sample["text"].lstrip()

    if not sample["tagged"]:
        return "texto_livre"

    if text.startswith("<pergunta>"):
        return "pergunta_resposta"

    if text.startswith("<topico>"):
        return "topico_conteudo"

    if text.startswith("<exercicio>"):
        return "exercicio_resolvido"

    return "outro"


def is_valid_sample(sample: dict) -> bool:
    """
    Filtro simples para evitar amostras quebradas.
    """
    if not sample["tagged"]:
        # Texto livre: só precisa ter conteúdo mínimo (já garantido no chunking).
        return len(sample["text"].strip()) >= FREE_TEXT_MIN_CHARS

    text = sample["text"].strip()

    if text.startswith("<pergunta>"):
        return "<resposta>" in text

    if text.startswith("<topico>"):
        return "<conteudo>" in text

    if text.startswith("<exercicio>"):
        return "<resolucao>" in text or "<gabarito>" in text

    return False


def split_samples_by_document(
    documents,
    train_ratio: float,
    val_ratio: float,
    test_ratio: float,
):
    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-6

    rng = random.Random()

    train_samples = []
    val_samples = []
    test_samples = []

    stats = []

    for doc in documents:
        doc_name = doc["name"]
        text = doc["text"]

        samples = split_semantic_samples(text)
        samples = [s for s in samples if is_valid_sample(s)]

        if not samples:
            stats.append({
                "documento": doc_name,
                "total": 0,
                "train": 0,
                "val": 0,
                "test": 0,
                "tipos": {}
            })
            continue

        rng.shuffle(samples)

        n = len(samples)

        n_train = int(n * train_ratio)
        n_val = int(n * val_ratio)

        if n >= 10:
            n_train = max(1, n_train)
            n_val = max(1, n_val)
            n_test = n - n_train - n_val

            if n_test < 1:
                n_train = max(1, n_train - 1)
                n_test = 1
        else:
            n_train = max(1, int(n * train_ratio))
            n_val = 1 if n >= 3 else 0
            n_test = n - n_train - n_val

        train_part = samples[:n_train]
        val_part = samples[n_train:n_train + n_val]
        test_part = samples[n_train + n_val:]

        train_samples.extend([
            {"documento": doc_name, "tipo": classify_sample(s), "text": s["text"]}
            for s in train_part
        ])

        val_samples.extend([
            {"documento": doc_name, "tipo": classify_sample(s), "text": s["text"]}
            for s in val_part
        ])

        test_samples.extend([
            {"documento": doc_name, "tipo": classify_sample(s), "text": s["text"]}
            for s in test_part
        ])

        stats.append({
            "documento": doc_name,
            "total": n,
            "train": len(train_part),
            "val": len(val_part),
            "test": len(test_part),
            "tipos": dict(Counter(classify_sample(s) for s in samples)),
        })

    rng.shuffle(train_samples)
    rng.shuffle(val_samples)
    rng.shuffle(test_samples)

    return train_samples, val_samples, test_samples, stats


train_samples, val_samples, test_samples, split_stats = split_samples_by_document(
    documents=documents,
    train_ratio=cfg.train_ratio,
    val_ratio=cfg.val_ratio,
    test_ratio=cfg.test_ratio
)

print("Resumo por documento:")
print("arquivo | total | train | val | test | tipos")

for item in split_stats:
    print(
        f"{item['documento']} | "
        f"{item['total']:,} | "
        f"{item['train']:,} | "
        f"{item['val']:,} | "
        f"{item['test']:,} | "
        f"{item['tipos']}"
    )

print("\nTotais:")
print("Amostras treino:", len(train_samples))
print("Amostras validação:", len(val_samples))
print("Amostras teste:", len(test_samples))

print("\nTipos no treino:", Counter(s["tipo"] for s in train_samples))
print("Tipos na validação:", Counter(s["tipo"] for s in val_samples))
print("Tipos no teste:", Counter(s["tipo"] for s in test_samples))

if not train_samples:
    raise ValueError("Nenhuma amostra de treino foi criada.")

if not val_samples:
    raise ValueError("Nenhuma amostra de validação foi criada.")

if not test_samples:
    print("Aviso: nenhuma amostra de teste foi criada. O corpus pode estar pequeno.")

Resumo por documento:
arquivo | total | train | val | test | tipos
teste.txt | 4 | 3 | 1 | 0 | {'texto_livre': 4}

Totais:
Amostras treino: 3
Amostras validação: 1
Amostras teste: 0

Tipos no treino: Counter({'texto_livre': 3})
Tipos na validação: Counter({'texto_livre': 1})
Tipos no teste: Counter()
Aviso: nenhuma amostra de teste foi criada. O corpus pode estar pequeno.


## 5. Dataset semântico com padding

In [7]:
def tokenize_sample(sample: str) -> torch.Tensor:
    ids = tokenizer.encode(sample).ids
    ids = [BOS_ID] + ids + [EOS_ID]
    return torch.tensor(ids, dtype=torch.long)


class SemanticCausalDataset(Dataset):
    def __init__(self, samples, block_size: int, is_train: bool = False):
        self.samples = samples
        self.block_size = block_size
        self.is_train = is_train

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        ids = tokenize_sample(sample["text"])

        if len(ids) > self.block_size + 1:
            max_start = len(ids) - self.block_size - 1

            if self.is_train:
                start = random.randint(0, max_start)
            else:
                # Para validação/teste, usa sempre o início para manter determinístico.
                start = 0

            ids = ids[start:start + self.block_size + 1]

        return ids


def causal_collate_fn(batch):
    max_len = max(len(ids) for ids in batch)
    max_len = min(max_len, cfg.block_size + 1)

    padded = torch.full(
        (len(batch), max_len),
        fill_value=PAD_ID,
        dtype=torch.long
    )

    for i, ids in enumerate(batch):
        ids = ids[:max_len]
        padded[i, :len(ids)] = ids

    x = padded[:, :-1]
    y = padded[:, 1:]

    return x, y


train_ds = SemanticCausalDataset(train_samples, cfg.block_size, is_train=True)
val_ds = SemanticCausalDataset(val_samples, cfg.block_size, is_train=False)
test_ds = SemanticCausalDataset(test_samples, cfg.block_size, is_train=False) if test_samples else None

train_loader = DataLoader(
    train_ds,
    batch_size=cfg.batch_size,
    shuffle=True,
    drop_last=True,
    collate_fn=causal_collate_fn,
)

val_loader = DataLoader(
    val_ds,
    batch_size=cfg.batch_size,
    shuffle=False,
    drop_last=False,
    collate_fn=causal_collate_fn,
)

if test_ds is not None:
    test_loader = DataLoader(
        test_ds,
        batch_size=cfg.batch_size,
        shuffle=False,
        drop_last=False,
        collate_fn=causal_collate_fn,
    )
else:
    test_loader = None

print("Batches treino:", len(train_loader))
print("Batches validação:", len(val_loader))
print("Batches teste:", len(test_loader) if test_loader else 0)

Batches treino: 0
Batches validação: 1
Batches teste: 0


## 6. Modelo

In [8]:
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        rms = x.pow(2).mean(dim=-1, keepdim=True)
        x = x * torch.rsqrt(rms + self.eps)
        return self.weight * x


class CausalSelfAttention(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()

        self.n_head = cfg.n_head
        self.head_dim = cfg.n_embd // cfg.n_head
        self.dropout = cfg.dropout

        self.qkv = nn.Linear(cfg.n_embd, 3 * cfg.n_embd, bias=False)
        self.proj = nn.Linear(cfg.n_embd, cfg.n_embd, bias=False)
        self.resid_dropout = nn.Dropout(cfg.dropout)

    def forward(self, x, kv_cache: Optional[dict] = None, use_cache: bool = False):
        B, T, C = x.shape

        qkv = self.qkv(x)
        q, k, v = qkv.chunk(3, dim=-1)

        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        if use_cache:
            if kv_cache is not None:
                if "k" in kv_cache and "v" in kv_cache:
                    k = torch.cat([kv_cache["k"], k], dim=2)
                    v = torch.cat([kv_cache["v"], v], dim=2)

                kv_cache["k"] = k
                kv_cache["v"] = v

            is_causal = False
        else:
            is_causal = True

        y = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=None,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=is_causal,
        )

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.proj(y)
        y = self.resid_dropout(y)
        return y


class MLP(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()

        hidden_dim = int(8 * cfg.n_embd / 3)

        self.w1 = nn.Linear(cfg.n_embd, hidden_dim, bias=False)
        self.w2 = nn.Linear(cfg.n_embd, hidden_dim, bias=False)
        self.w3 = nn.Linear(hidden_dim, cfg.n_embd, bias=False)
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, x):
        x = self.w3(self.w1(x) * F.silu(self.w2(x)))
        return self.dropout(x)


class DecoderBlock(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.norm1 = RMSNorm(cfg.n_embd)
        self.attn = CausalSelfAttention(cfg)
        self.norm2 = RMSNorm(cfg.n_embd)
        self.mlp = MLP(cfg)

    def forward(self, x, kv_cache: Optional[dict] = None, use_cache: bool = False):
        x = x + self.attn(self.norm1(x), kv_cache=kv_cache, use_cache=use_cache)
        x = x + self.mlp(self.norm2(x))
        return x

## 7. Transformer decoder-only

In [9]:
class DecoderOnlyTransformer(nn.Module):
    def __init__(self, cfg: Config, vocab_size: int):
        super().__init__()

        self.cfg = cfg
        self.vocab_size = vocab_size

        self.token_emb = nn.Embedding(vocab_size, cfg.n_embd)
        self.pos_emb = nn.Embedding(cfg.block_size, cfg.n_embd)
        self.drop = nn.Dropout(cfg.dropout)

        self.blocks = nn.ModuleList([DecoderBlock(cfg) for _ in range(cfg.n_layer)])
        self.norm_f = RMSNorm(cfg.n_embd)
        self.lm_head = nn.Linear(cfg.n_embd, vocab_size, bias=False)

        self.lm_head.weight = self.token_emb.weight

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

            if module.bias is not None:
                nn.init.zeros_(module.bias)

        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None, kv_caches: Optional[list] = None, use_cache: bool = False):
        B, T = idx.shape

        if use_cache:
            past_len = 0

            if kv_caches and kv_caches[0] is not None and "k" in kv_caches[0]:
                past_len = kv_caches[0]["k"].shape[2]

            if past_len + T > self.cfg.block_size:
                raise ValueError(
                    f"Contexto excedeu block_size={self.cfg.block_size}. "
                    "Reduza o prompt ou aumente block_size."
                )

            pos = torch.arange(past_len, past_len + T, device=idx.device)
        else:
            if T > self.cfg.block_size:
                raise ValueError(f"Sequência de tamanho {T} excede block_size={self.cfg.block_size}.")

            pos = torch.arange(0, T, device=idx.device)

        tok = self.token_emb(idx)
        pos = self.pos_emb(pos)[None, :, :]
        x = self.drop(tok + pos)

        if kv_caches is None:
            kv_caches = [None] * len(self.blocks)

        for block, cache in zip(self.blocks, kv_caches):
            x = block(x, kv_cache=cache, use_cache=use_cache)

        x = self.norm_f(x)
        logits = self.lm_head(x)

        loss = None

        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1),
                ignore_index=PAD_ID,
            )

        return logits, loss

    @torch.no_grad()
    def generate(
        self,
        prompt_ids: list[int],
        max_new_tokens: int = 150,
        temperature: float = 0.7,
        top_k: int = 50,
    ):
        self.eval()

        max_prompt_len = max(1, self.cfg.block_size - max_new_tokens)
        prompt_ids = prompt_ids[-max_prompt_len:]

        idx = torch.tensor(
            [prompt_ids],
            dtype=torch.long,
            device=next(self.parameters()).device
        )

        kv_caches = [dict() for _ in self.blocks]
        generated = prompt_ids[:]

        logits, _ = self(idx, kv_caches=kv_caches, use_cache=True)
        next_logits = logits[:, -1, :]

        for _ in range(max_new_tokens):
            logits_step = next_logits / max(temperature, 1e-6)

            if top_k is not None:
                k = min(top_k, logits_step.size(-1))
                values, _ = torch.topk(logits_step, k=k)
                cutoff = values[:, [-1]]
                logits_step = torch.where(
                    logits_step < cutoff,
                    torch.full_like(logits_step, float("-inf")),
                    logits_step,
                )

            probs = F.softmax(logits_step, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)

            token_id = int(next_id.item())
            generated.append(token_id)

            if token_id == EOS_ID:
                break

            if len(generated) >= self.cfg.block_size:
                break

            logits, _ = self(next_id, kv_caches=kv_caches, use_cache=True)
            next_logits = logits[:, -1, :]

        return generated

## 8. Inicializar modelo e otimizador

In [10]:
model = DecoderOnlyTransformer(cfg, vocab_size).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Parâmetros: {n_params / 1e6:.2f}M")

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=cfg.learning_rate,
    weight_decay=cfg.weight_decay,
    betas=(0.9, 0.95),
)

Parâmetros: 4.36M


## 9. Avaliação em treino, validação e teste

In [11]:
@torch.no_grad()
def evaluate_loader(loader, max_batches: int | None = None):
    if loader is None:
        return float("nan")

    model.eval()
    losses = []

    for i, (x, y) in enumerate(loader):
        if max_batches is not None and i >= max_batches:
            break

        x = x.to(device)
        y = y.to(device)

        _, loss = model(x, y)
        losses.append(loss.item())

    model.train()

    if not losses:
        return float("nan")

    return sum(losses) / len(losses)


@torch.no_grad()
def estimate_loss():
    return {
        "train": evaluate_loader(train_loader, cfg.eval_batches),
        "val": evaluate_loader(val_loader, cfg.eval_batches),
        "test": evaluate_loader(test_loader, cfg.eval_batches),
    }


initial_losses = estimate_loss()
print("Loss inicial:", initial_losses)

Loss inicial: {'train': nan, 'val': 8.467363357543945, 'test': nan}


## 10. Loop de treinamento

In [ ]:
checkpoint_dir = Path(cfg.checkpoint_dir)
checkpoint_dir.mkdir(parents=True, exist_ok=True)

model.train()
step = 0
best_val_loss = float("inf")

pbar = tqdm(total=cfg.max_steps, desc="Treinando")

while step < cfg.max_steps:
    for x, y in train_loader:
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad(set_to_none=True)

        _, loss = model(x, y)
        loss.backward()

        if cfg.grad_clip is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)

        optimizer.step()

        step += 1
        pbar.update(1)
        pbar.set_postfix(loss=float(loss.item()))

        if step % cfg.eval_interval == 0 or step == 1:
            losses = estimate_loss()

            print(
                f"step {step}: "
                f"train loss={losses['train']:.4f}, "
                f"val loss={losses['val']:.4f}, "
                f"test loss={losses['test']:.4f}"
            )

            ckpt = {
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "config": cfg.__dict__,
                "vocab_size": vocab_size,
                "step": step,
                "losses": losses,
            }

            torch.save(ckpt, checkpoint_dir / "decoder_only_musica_swiglu_semantic_last.pt")

            if losses["val"] < best_val_loss:
                best_val_loss = losses["val"]
                torch.save(ckpt, checkpoint_dir / "decoder_only_musica_swiglu_semantic_best.pt")
                print(f"Novo melhor checkpoint salvo. val loss={best_val_loss:.4f}")

        if step >= cfg.max_steps:
            break

pbar.close()
print("Treinamento finalizado.")

Treinando:   0%|          | 0/400 [00:00<?, ?it/s]

## 11. Avaliação final completa

In [ ]:
final_train_loss = evaluate_loader(train_loader, max_batches=None)
final_val_loss = evaluate_loader(val_loader, max_batches=None)
final_test_loss = evaluate_loader(test_loader, max_batches=None)

print(f"Final train loss: {final_train_loss:.4f}")
print(f"Final val loss:   {final_val_loss:.4f}")
print(f"Final test loss:  {final_test_loss:.4f}")

## 12. Inferência

In [ ]:
def encode_prompt(text: str) -> list[int]:
    return [BOS_ID] + tokenizer.encode(text).ids


def decode_output(ids: list[int]) -> str:
    ids = [i for i in ids if i not in {PAD_ID, BOS_ID, EOS_ID}]
    return tokenizer.decode(ids)


prompt = '''
o que é andamento?
'''

prompt_ids = encode_prompt(prompt)

generated_ids = model.generate(
    prompt_ids=prompt_ids,
    max_new_tokens=150,
    temperature=0.3,
    top_k=50,
)

print(decode_output(generated_ids))

## 13. Avaliação qualitativa por prompts fixos

In [ ]:
@torch.no_grad()
def avaliar_prompts(model, prompts, max_new_tokens=160, temperature=0.3, top_k=50):
    model.eval()

    for i, prompt in enumerate(prompts, start=1):
        prompt_ids = encode_prompt(prompt)

        generated_ids = model.generate(
            prompt_ids=prompt_ids,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_k=top_k,
        )

        saida = decode_output(generated_ids)

        print("=" * 80)
        print(f"AVALIAÇÃO {i}")
        print("=" * 80)
        print(saida)
        print()


prompts_avaliacao = [
    '''<pergunta>
        O que é campo harmônico?
    <resposta>
    ''',

    '''<pergunta>
        Monte o campo harmônico maior de Dó.
    <resposta>
    ''',

    '''<pergunta>
        Monte o campo harmônico maior de Fá.
    <resposta>
    ''',

    '''<pergunta>
        Por que em Fá maior usamos Bb e não A#?
    <resposta>
    ''',

    '''<exercicio>
        Monte o ii-V-I em Ré maior.
    <resolucao>
    ''',

    '''<topico>
        Dominantes secundários
    <conteudo>
    ''',
]

avaliar_prompts(model, prompts_avaliacao)

## 14. Carregar melhor checkpoint

In [ ]:
ckpt_path = Path(cfg.checkpoint_dir) / "decoder_only_musica_swiglu_semantic_best.pt"

if ckpt_path.exists():
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    print("Melhor checkpoint carregado.")
    print("Step:", ckpt.get("step"))
    print("Losses:", ckpt.get("losses"))
else:
    print("Nenhum checkpoint encontrado.")